# Prerequisites

In [8]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

--2026-09-24 13:35:11--  https://www.gutenberg.org/ebooks/103.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/103/pg103.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-24 13:35:11--  https://www.gutenberg.org/cache/epub/103/pg103.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 403712 (394K) [text/plain]
Saving to: ‘around_the_world_in_80_days.txt’

around_the_world_in 100%[===================>] 394.25K  1.18MB/s    in 0.3s    

2026-09-24 13:35:11 (1.18 MB/s) - ‘around_the_world_in_80_days.txt’ saved [403712/403712]



# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [9]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [10]:
# Defind the rdd
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [11]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [12]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [13]:
# Note and explain the output of the below command
words

PythonRDD[7] at RDD at PythonRDD.scala:59

The “words” display does not show the split words, but rather the signature of the RDD object (PythonRDD). PySpark uses lazy evaluation: the `flatMap` transformation is indeed recorded in the execution plan, but the computation will not be launched on the cluster until an “action” is requested.

<ADD EXPLANATION HERE>

In [14]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'Project',
 'Gutenberg',
 'License',
 'included',
 'with',
 'this',
 'eBook',
 'or',
 'online',
 'at',
 'www.gutenberg.org.',
 'If',
 'you',
 'are',
 'not',
 'located',
 'in',
 'the',
 'United',
 'States,',
 'you',
 'will',
 'have',
 'to',
 'check',
 'the',
 'laws',
 'of',
 'the',
 'country',
 'where',
 'you',
 'are',
 'located',
 'before',
 'using',
 'this',
 'eBook.',
 '',
 'Title:',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 'Author:',
 'Jules'

Unlike the previous command, `collect()` is an action. It forces the computation plan to run immediately. Spark will distribute the task, extract all the words from the distributed text, and then bring all the results back into the local memory of our machine (the driver node) in the form of a Python list.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

The `flatMap` operation combines two actions: first, it applies the function (in this case, `split(‘ ’)`, which transforms a line into 
a list of words), and then it “flattens” all these nested lists into a single, large dimension. The result is an RDD where each element 
is an individual word, rather than a list of words.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [21]:
# a. count the occurence of each word
word_counts = words.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

print("Comptage :", word_counts.take(5))

Comptage : [('Gutenberg', 60), ('eBook', 6), ('of', 1875), ('Around', 4), ('', 2193)]


In [22]:
# b. a common first step in text analysis, change all capital letters to lower case
words_lower = rdd.flatMap(lambda line: line.split(' ')).map(lambda word: word.lower())
print("Minuscules :", words_lower.take(5))

Minuscules : ['the', 'project', 'gutenberg', 'ebook', 'of']


In [27]:
# c. eliminate the stop words.
stop_words = {"the", "and", "of", "to", "a", "in", "that", "is", "was", "he", "for", "it", "with", "as", "his", "on", "be", "at", "by", "i", "this", "had", "not", "are", "but"}
words_filtered = words_lower.filter(lambda word: word not in stop_words and word != "")
print("Sans stop words :", words_filtered.take(5))

Sans stop words : ['project', 'gutenberg', 'ebook', 'around', 'world']


In [31]:
# d. sort in alphabetical order
clean_counts = words_filtered.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

sorted_alpha = clean_counts.sortByKey(ascending=True)
print("Ordre alphabétique :", sorted_alpha.take(5))

Ordre alphabétique : [('#103]', 1), ('#516,', 1), ('$5,000)', 1), ('&c.,', 1), ('($1', 1)]


In [32]:
# e. sort descending by word frequency
sorted_freq = clean_counts.sortBy(lambda x: x[1], ascending=False)
print("Plus fréquents :", sorted_freq.take(5))

Plus fréquents : [('which', 490), ('mr.', 373), ('fogg', 365), ('from', 323), ('were', 303)]


In [34]:
# f. remove punctuations and blank spaces
words_clean_freq = (words_filtered
    .map(lambda word: word.strip(string.punctuation + ' '))
    .filter(lambda word: word != '')
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print("Mots nettoyés les plus fréquents :", words_clean_freq.take(5))

Mots nettoyés les plus fréquents : [('fogg', 577), ('which', 515), ('passepartout', 392), ('mr', 373), ('from', 324)]


# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [35]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
    # Convert (name, age) to (name, (age, 1)) to count the occurrences
  .map(lambda x: (x[0], (x[1], 1)))
  # Adds the ages and counters for each key (common name)
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # Divide the sum of the ages by the total number to find the average
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

print("Moyennes d'âge :", agesRDD.collect())

Moyennes d'âge : [('Brooke', 22.5), ('Denny', 31.0), ('TD', 35.0), ('Jules', 30.0)]


## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [38]:
import time
import string

def optimized_pipeline(raw_rdd):
    stop_words = {"the", "and", "of", "to", "a", "in", "that", "is", "was", "he", "for", "it", "with", "as", "his", "on", "be", "at", "by", "i", "this", "had", "not", "are", "but", "from", "or"}
    return (raw_rdd
        .flatMap(lambda line: line.split(' '))
        .map(lambda word: word.lower().strip(string.punctuation + ' '))
        .filter(lambda word: word != '' and word not in stop_words)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda a, b: a + b)
        .sortBy(lambda x: x[1], ascending=False)
    )

def time_rdd_execution(pipeline_func, data_rdd):
    start_time = time.time()
    unique_words_count = pipeline_func(data_rdd).count()
    end_time = time.time()
    
    print(f"Temps d'exécution : {end_time - start_time:.4f} secondes")
    print(f"Mots uniques traités : {unique_words_count}")

time_rdd_execution(optimized_pipeline, rdd)

Temps d'exécution : 0.5020 secondes
Mots uniques traités : 8541


## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [39]:
!wget -nc -O tour_du_monde.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

rdd_fr = sc.textFile('tour_du_monde.txt')
# Adaptation des stopwords et ponctuation pour la version française
stop_words_fr = {"le", "la", "les", "l", "de", "d", "des", "un", "une", "et", "à", "en", "il", "est", "dans", "pour", "que", "qu", "qui", "ne", "pas", "se", "s", "sur", "son", "sa", "au", "a", "ce", "plus", "par", "c", "n"}

def nlp_pipeline_fr(raw_rdd):
    return (raw_rdd
        .flatMap(lambda line: line.split(' '))
        .map(lambda word: word.lower().strip(string.punctuation + ' «»\u2028\u2029'))
        .filter(lambda word: word != '' and word not in stop_words_fr)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda x, y: x + y)
        .sortBy(lambda x: x[1], ascending=False)
    )

top_en = optimized_pipeline(rdd).take(10)
top_fr = nlp_pipeline_fr(rdd_fr).take(10)

print("Top 10 mots (Anglais)  :", top_en)
print("Top 10 mots (Français) :", top_fr)

--2026-09-24 14:10:23--  https://www.gutenberg.org/ebooks/46541.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/46541/pg46541.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-24 14:10:23--  https://www.gutenberg.org/cache/epub/46541/pg46541.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 472731 (462K) [text/plain]
Saving to: ‘tour_du_monde.txt’

tour_du_monde.txt   100%[===================>] 461.65K   903KB/s    in 0.5s    

2026-09-24 14:10:24 (903 KB/s) - ‘tour_du_monde.txt’ saved [472731/472731]

Top 10 mots (Anglais)  : [('fogg', 577), ('which', 515), ('passepartout', 392), ('mr', 373), ('you', 315), ('him', 314), ('were', 307), ('would', 278), ('have', 270), ('phileas', 250)]
Top 10 mots (